# Day 44：MCP 与 Agent 生态（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-044` ｜ **开发分支**：`feat/day44-mcp-jsonrpc-and-memory`  
> **配套 HTML5 课件**：`courseware/day44/index.html` ｜ **独立版本快照**：`versions/day44/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天我们聊一个 2025~2026 年整个 AI 工程圈最火的协议——**MCP（Model Context Protocol）**。Anthropic 提出 MCP 的目的特别纯粹：以前 M 个 Agent 框架要对接 N 个企业数据源，需要写 M×N 套胶水代码；有了 MCP 这根“AI 世界的 Type-C 接口”，数据源只需写一次 **MCP Server**，不管是 Claude、Cursor 还是我们星辰智脑的 LangGraph Agent，插上就能自动发现并调用工具！今天上半场老陈带你按 **JSON-RPC 2.0** 规范亲手撸一套 MCP Server + Client；下半场我们补齐 Agent 的另一块核心拼图——**短期滑动记忆 + SQLite 长期记忆**。

### 承上启下
- **昨日基线**：昨天（Day 43）我们组建了多 Agent 行业研究团队，解决了复杂任务的角色拆分问题。但在接入公司内部的 Jira、CRM、GitLab、企业微信等十几个系统时，阿杰抱怨道：难道每个 Agent 项目都要把这些工具重新封装一遍吗？
- **明日预告**：明天（Day 45）我们将迎来第六周综合实战周测，同时拆解 Dify / Coze 低代码平台的 DSL 工作流引擎，帮你建立“什么时候用低代码、什么时候必须代码自研”的架构选型矩阵。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day44'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-044）

星辰数科内部已有 3 个不同业务线的 Agent 团队，分别重复编写了访问员工通讯录、查询工单系统和读取制度文档的工具代码，接口变更时三边同时报错。同时，员工反馈每次新开对话都要重复介绍自己的部门、常用汇报风格，缺乏跨会话记忆。

**今日迭代目标：**
- 掌握 MCP 架构（Host / Client / Server）及底层 JSON-RPC 2.0 请求-响应-错误码规范。
- 零外部端口依赖，纯 Python 实现标准 JSON-RPC 2.0 的 `MCPServer` 与 `MCPClient`，支持 `initialize`、`tools/list`、`tools/call`、`resources/list`、`resources/read`。
- 实现 Agent 双层记忆系统：`ShortTermMemory`（滑动窗口防上下文溢出）+ `SQLiteLongTermMemory`（跨会话事实与偏好持久化与相关性召回）。

**验收标准：**
- MCP Client 与 MCP Server 握手后能动态发现工具 Schema、调用工具并返回符合 JSON-RPC 2.0 结构的响应。
- 调用未知 JSON-RPC 方法时返回标准错误码 `-32601`（Method not found），参数非法时返回 `-32602`。
- 长期记忆落库 SQLite 后，新建记忆实例仍能按 `user_id` 和关键词召回历史偏好与事实，所有单测本地 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. JSON-RPC 2.0 报文结构与标准错误码

每一条合法的 JSON-RPC 2.0 请求必须包含 `jsonrpc: '2.0'`、`id`、`method` 和可选 `params`；响应必须包含 `jsonrpc: '2.0'`、`id` 以及 `result` 或 `error`（二者互斥）。常见标准错误码：
- `-32700`: Parse error（JSON 解析失败）
- `-32600`: Invalid Request（请求体缺少必填字段）
- `-32601`: Method not found（方法不存在）
- `-32602`: Invalid params（工具参数校验失败）
- `-32603`: Internal error（服务端内部执行异常）

> **⚠️ 老陈避坑**：坑点：使用 `stdio` 传输 MCP 报文时，Server 端千万不能随手用 `print('debug...')` 往 stdout 打调试日志！stdout 是 JSON-RPC 协议专用通道，任何非 JSON 杂音都会让 Client 直接解析崩溃，日志必须统一输出到 `stderr`。

### 2. 2. MCP 中 Tools 与 Resources 的职责区分

很多新手分不清什么时候该暴露成 `Tool`，什么时候暴露成 `Resource`：
- **Tools（模型控制）**：有计算、检索或副作用的动作（如查库存、发邮件、跑 SQL），由大模型主动决定何时调用；
- **Resources（应用控制）**：类似只读文件或 REST GET 端点（如 `starmind://docs/travel-policy`），由宿主应用提前读取并附着到上下文中。

> **⚠️ 老陈避坑**：坑点：不要把敏感写操作（如删库、转账）伪装成 Resource，也不要在未鉴权的情况下把内网全量数据库直接暴露给公共 MCP Server。

### 3. 3. 长期记忆的“遗忘与更新”治理

长期记忆不是只增不减的垃圾桶！当员工从“研发部”转岗到“解决方案部”时，旧记忆必须支持覆盖更新或按 `importance` + `updated_at` 衰减排序，否则 Agent 半年后还会按旧部门发审批。

> **⚠️ 老陈避坑**：坑点：把用户的每一句闲聊（比如“今天杭州下雨了”）都存进长期记忆，会导致记忆库迅速噪音化、召回准确率暴跌。必须设置显式触发规则或重要性阈值过滤。


## 💻 实操步骤 1：步骤一：严格遵循 JSON-RPC 2.0 的自研 MCP Server 与 MCP Client

- **目标文件**：`versions/day44/src/day44_mcp_protocol.py`
- **老陈代码旁白**：我们用纯标准库实现一套完全符合 JSON-RPC 2.0 与 MCP 核心原语的 `MCPServer` 和 `MCPClient`。Client 和 Server 之间通过标准 JSON 字符串收发消息，既可以在本地内存零端口直连测试，也可以直接套上 `sys.stdin/stdout` 变成真实的 Stdio MCP Server！


In [ ]:
"""Day 44 - 步骤一：严格遵循 JSON-RPC 2.0 规范的自研 MCP Server 与 MCP Client。"""

from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional

JSONRPC_VERSION = "2.0"
ERR_PARSE = -32700
ERR_INVALID_REQUEST = -32600
ERR_METHOD_NOT_FOUND = -32601
ERR_INVALID_PARAMS = -32602
ERR_INTERNAL = -32603


@dataclass
class MCPToolSpec:
    name: str
    description: str
    input_schema: Dict[str, Any]
    handler: Callable[[Dict[str, Any]], Any]


@dataclass
class MCPResourceSpec:
    uri: str
    name: str
    mime_type: str
    content: str


class MCPServer:
    """轻量级企业 MCP Server：暴露 Tools 与 Resources，并严格按 JSON-RPC 2.0 响应。"""

    def __init__(self, server_name: str, version: str = "1.0.0") -> None:
        self.server_name = server_name
        self.version = version
        self._tools: Dict[str, MCPToolSpec] = {}
        self._resources: Dict[str, MCPResourceSpec] = {}

    def register_tool(
        self,
        name: str,
        description: str,
        input_schema: Dict[str, Any],
        handler: Callable[[Dict[str, Any]], Any],
    ) -> None:
        self._tools[name] = MCPToolSpec(name, description, input_schema, handler)

    def register_resource(self, uri: str, name: str, content: str, mime_type: str = "text/plain") -> None:
        self._resources[uri] = MCPResourceSpec(uri, name, mime_type, content)

    def _ok(self, req_id: Any, result: Any) -> str:
        return json.dumps({"jsonrpc": JSONRPC_VERSION, "id": req_id, "result": result}, ensure_ascii=False)

    def _err(self, req_id: Any, code: int, message: str) -> str:
        return json.dumps(
            {"jsonrpc": JSONRPC_VERSION, "id": req_id, "error": {"code": code, "message": message}},
            ensure_ascii=False,
        )

    def handle_json_line(self, raw_line: str) -> str:
        """处理单条 JSON-RPC 2.0 请求报文并返回响应 JSON 字符串。"""
        try:
            payload = json.loads(raw_line)
        except json.JSONDecodeError as exc:
            return self._err(None, ERR_PARSE, f"JSON 解析失败: {exc}")

        if not isinstance(payload, dict) or payload.get("jsonrpc") != JSONRPC_VERSION or "method" not in payload:
            return self._err(payload.get("id") if isinstance(payload, dict) else None, ERR_INVALID_REQUEST, "非法 JSON-RPC 2.0 请求")

        req_id = payload.get("id")
        method = payload["method"]
        params = payload.get("params") or {}

        if method == "initialize":
            return self._ok(
                req_id,
                {
                    "protocolVersion": "2025-03-26",
                    "serverInfo": {"name": self.server_name, "version": self.version},
                    "capabilities": {"tools": {}, "resources": {}},
                },
            )

        if method == "tools/list":
            tools = [
                {"name": t.name, "description": t.description, "inputSchema": t.input_schema}
                for t in self._tools.values()
            ]
            return self._ok(req_id, {"tools": tools})

        if method == "tools/call":
            tool_name = params.get("name")
            arguments = params.get("arguments", {})
            if tool_name not in self._tools:
                return self._err(req_id, ERR_INVALID_PARAMS, f"未知工具: {tool_name}")
            spec = self._tools[tool_name]
            required = spec.input_schema.get("required", [])
            missing = [k for k in required if k not in arguments]
            if missing:
                return self._err(req_id, ERR_INVALID_PARAMS, f"缺少必填参数: {missing}")
            try:
                output = spec.handler(arguments)
                return self._ok(
                    req_id,
                    {
                        "content": [{"type": "text", "text": json.dumps(output, ensure_ascii=False)}],
                        "isError": False,
                    },
                )
            except Exception as exc:
                return self._ok(
                    req_id,
                    {"content": [{"type": "text", "text": f"工具执行失败: {exc}"}], "isError": True},
                )

        if method == "resources/list":
            resources = [
                {"uri": r.uri, "name": r.name, "mimeType": r.mime_type}
                for r in self._resources.values()
            ]
            return self._ok(req_id, {"resources": resources})

        if method == "resources/read":
            uri = params.get("uri")
            if uri not in self._resources:
                return self._err(req_id, ERR_INVALID_PARAMS, f"资源不存在: {uri}")
            r = self._resources[uri]
            return self._ok(req_id, {"contents": [{"uri": r.uri, "mimeType": r.mime_type, "text": r.content}]})

        return self._err(req_id, ERR_METHOD_NOT_FOUND, f"不支持的方法: {method}")


class MCPClient:
    """MCP 客户端：封装 JSON-RPC 2.0 请求编号、握手与能力调用。"""

    def __init__(self, transport_fn: Callable[[str], str]) -> None:
        self._transport = transport_fn
        self._seq = 0

    def request(self, method: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        self._seq += 1
        req = {"jsonrpc": JSONRPC_VERSION, "id": self._seq, "method": method, "params": params or {}}
        raw_resp = self._transport(json.dumps(req, ensure_ascii=False))
        return json.loads(raw_resp)

    def initialize(self) -> Dict[str, Any]:
        return self.request("initialize", {"clientInfo": {"name": "starmind-host", "version": "1.0"}})

    def list_tools(self) -> List[Dict[str, Any]]:
        resp = self.request("tools/list")
        return resp["result"]["tools"]

    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        return self.request("tools/call", {"name": name, "arguments": arguments})

    def read_resource(self, uri: str) -> str:
        resp = self.request("resources/read", {"uri": uri})
        return resp["result"]["contents"][0]["text"]


def create_starmind_enterprise_mcp_server() -> MCPServer:
    """构建星辰数科企业内网演示 MCP Server。"""
    server = MCPServer("starmind-enterprise-mcp", version="1.2.0")
    directory = {
        "老陈": {"dept": "AI 平台架构部", "title": "首席架构师", "email": "chen@starmind.cn"},
        "林产品": {"dept": "智能产品部", "title": "高级产品经理", "email": "lin@starmind.cn"},
    }
    server.register_tool(
        name="lookup_employee",
        description="按员工姓名查询所属部门、职级与企业邮箱",
        input_schema={
            "type": "object",
            "properties": {"name": {"type": "string"}},
            "required": ["name"],
        },
        handler=lambda args: directory.get(args["name"], {"found": False, "name": args["name"]}),
    )
    server.register_resource(
        uri="starmind://policies/mcp-security",
        name="星辰数科 MCP 安全接入规范",
        content="所有生产 MCP Server 必须启用最小权限白名单，且禁止向 stdout 打印非 JSON-RPC 调试信息。",
    )
    return server


if __name__ == "__main__":
    srv = create_starmind_enterprise_mcp_server()
    client = MCPClient(transport_fn=srv.handle_json_line)
    print("握手响应:", client.initialize()["result"]["serverInfo"])
    print("发现工具:", [t["name"] for t in client.list_tools()])
    print("调用工具:", client.call_tool("lookup_employee", {"name": "老陈"})["result"])
    print("读取资源:", client.read_resource("starmind://policies/mcp-security"))


### 🔍 步骤 1 代码深度解析

1. `MCPServer.handle_json_line` 完整实现了 JSON-RPC 2.0 报文校验、标准错误码返回以及 MCP 五大核心方法。
2. `MCPClient` 通过注入 `transport_fn` 解耦了底层传输协议：本地单测直接传入 `srv.handle_json_line`，生产环境替换为 `subprocess` 的 `stdin/stdout` 或 HTTP POST 即可无缝切换。
3. 工具执行内部异常不会弄崩协议层，而是包装为 MCP 标准 `isError: True` 返回给大模型自主决策。

**预期运行输出参考：**
```text
握手响应: {'name': 'starmind-enterprise-mcp', 'version': '1.2.0'}
发现工具: ['lookup_employee']
调用工具: {'content': [{'type': 'text', 'text': '{"dept": "AI 平台架构部", "title": "首席架构师", "email": "chen@starmind.cn"}'}], 'isError': False}
读取资源: 所有生产 MCP Server 必须启用最小权限白名单，且禁止向 stdout 打印非 JSON-RPC 调试信息。
```


## 💻 实操步骤 2：步骤二：实现 Agent 短期滑动窗口记忆 + SQLite 跨会话长期记忆

- **目标文件**：`versions/day44/src/day44_agent_memory.py`
- **老陈代码旁白**：现在给我们的 Agent 装上“海马体”：`ShortTermMemory` 保证单次会话的滑动窗口不超长；`SQLiteLongTermMemory` 把用户的偏好和关键业务事实持久化到数据库，支持跨会话按关键词与重要性综合打分召回。


In [ ]:
"""Day 44 - 步骤二：Agent 短期滑动窗口记忆 + SQLite 跨会话长期记忆引擎。"""

from __future__ import annotations

import sqlite3
import time
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Deque, Dict, List, Tuple, Union


@dataclass
class MemoryItem:
    memory_id: int
    user_id: str
    category: str  # "preference" | "fact" | "episodic"
    content: str
    importance: int  # 1 ~ 5
    updated_at: float


class ShortTermMemory:
    """短期工作记忆：滑动窗口保留最近 max_turns 轮对话，超限时自动滚动压缩旧摘要。"""

    def __init__(self, max_messages: int = 6) -> None:
        self.max_messages = max_messages
        self._messages: Deque[Dict[str, str]] = deque()
        self.evicted_summary: List[str] = []

    def add(self, role: str, content: str) -> None:
        self._messages.append({"role": role, "content": content})
        while len(self._messages) > self.max_messages:
            old = self._messages.popleft()
            self.evicted_summary.append(f"{old['role']}: {old['content'][:24]}")

    def get_messages(self) -> List[Dict[str, str]]:
        return list(self._messages)


class SQLiteLongTermMemory:
    """基于 SQLite 的跨会话长期记忆库：支持写入、去重更新与相关性+重要性混合召回。"""

    def __init__(self, db_path: Union[str, Path] = ":memory:") -> None:
        self._conn = sqlite3.connect(str(db_path))
        self._conn.execute(
            """
            CREATE TABLE IF NOT EXISTS long_term_memories (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id TEXT NOT NULL,
                category TEXT NOT NULL,
                content TEXT NOT NULL,
                importance INTEGER NOT NULL,
                updated_at REAL NOT NULL,
                UNIQUE(user_id, category, content)
            )
            """
        )
        self._conn.commit()

    def remember(self, user_id: str, category: str, content: str, importance: int = 3) -> None:
        imp = max(1, min(5, int(importance)))
        now = time.time()
        self._conn.execute(
            """
            INSERT INTO long_term_memories (user_id, category, content, importance, updated_at)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(user_id, category, content)
            DO UPDATE SET importance = excluded.importance, updated_at = excluded.updated_at
            """,
            (user_id, category, content, imp, now),
        )
        self._conn.commit()

    def recall(self, user_id: str, query: str, top_k: int = 3) -> List[MemoryItem]:
        rows = self._conn.execute(
            "SELECT id, user_id, category, content, importance, updated_at FROM long_term_memories WHERE user_id = ?",
            (user_id,),
        ).fetchall()
        query_chars = {ch for ch in query.lower() if ch.strip()}
        scored: List[Tuple[float, MemoryItem]] = []
        for row in rows:
            item = MemoryItem(*row)
            content_chars = {ch for ch in item.content.lower() if ch.strip()}
            overlap = len(query_chars & content_chars) / max(len(query_chars), 1)
            # 综合打分：字符相关性 + 重要性加权
            score = overlap * 2.0 + item.importance * 0.4
            scored.append((score, item))
        scored.sort(key=lambda pair: (pair[0], pair[1].updated_at), reverse=True)
        return [item for _, item in scored[:top_k]]

    def close(self) -> None:
        self._conn.close()


class HybridMemoryManager:
    """双层记忆统一门面：拼装给大模型推理的完整记忆上下文。"""

    def __init__(self, user_id: str, long_term: SQLiteLongTermMemory, max_short_messages: int = 6) -> None:
        self.user_id = user_id
        self.short_term = ShortTermMemory(max_messages=max_short_messages)
        self.long_term = long_term

    def build_prompt_context(self, current_query: str, top_k: int = 2) -> Dict[str, object]:
        recalled = self.long_term.recall(self.user_id, current_query, top_k=top_k)
        return {
            "user_id": self.user_id,
            "long_term_memories": [f"[{m.category}/P{m.importance}] {m.content}" for m in recalled],
            "short_term_messages": self.short_term.get_messages(),
            "evicted_count": len(self.short_term.evicted_summary),
        }


if __name__ == "__main__":
    ltm = SQLiteLongTermMemory(":memory:")
    ltm.remember("u_chen", "preference", "汇报请始终使用三点式精简结论，不要长篇大论", importance=5)
    ltm.remember("u_chen", "fact", "负责杭州星辰数科 AI 平台架构团队", importance=4)
    ltm.remember("u_chen", "episodic", "上周五讨论过 MCP Server 安全接入规范", importance=3)

    mgr = HybridMemoryManager("u_chen", ltm, max_short_messages=2)
    mgr.short_term.add("user", "你好")
    mgr.short_term.add("assistant", "你好老陈，今天需要什么协助？")
    mgr.short_term.add("user", "帮我总结一下 MCP 接入汇报要点")

    ctx = mgr.build_prompt_context("MCP 汇报要点", top_k=2)
    print("组装后的双层记忆上下文:", ctx)
    ltm.close()


### 🔍 步骤 2 代码深度解析

1. `ShortTermMemory` 利用 `deque` 精确控制窗口大小，被挤出的历史消息不会静默丢失，而是进入 `evicted_summary` 便于生成滚动摘要。
2. `SQLiteLongTermMemory` 通过 `UNIQUE(user_id, category, content)` 的 `ON CONFLICT DO UPDATE` 实现幂等写入，避免重复记忆刷屏。
3. `recall` 将文本相关性与重要性权重 `importance` 结合，确保高优先级用户偏好（如汇报格式要求）稳定浮出水面。

**预期运行输出参考：**
```text
组装后的双层记忆上下文: {'user_id': 'u_chen', 'long_term_memories': ['[preference/P5] 汇报请始终使用三点式精简结论，不要长篇大论', '[episodic/P3] 上周五讨论过 MCP Server 安全接入规范'], 'short_term_messages': [{'role': 'assistant', 'content': '你好老陈，今天需要什么协助？'}, {'role': 'user', 'content': '帮我总结一下 MCP 接入汇报要点'}], 'evicted_count': 1}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day44/tests/test_day44.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 44 自动化单元测试：验证 JSON-RPC 2.0 MCP Server/Client 与 SQLite 双层记忆引擎。"""

import json
from pathlib import Path
from src.day44_mcp_protocol import (
    ERR_INVALID_PARAMS,
    ERR_METHOD_NOT_FOUND,
    MCPClient,
    create_starmind_enterprise_mcp_server,
)
from src.day44_agent_memory import HybridMemoryManager, SQLiteLongTermMemory


def test_mcp_handshake_tools_and_resources():
    srv = create_starmind_enterprise_mcp_server()
    cli = MCPClient(srv.handle_json_line)

    init = cli.initialize()
    assert init["jsonrpc"] == "2.0"
    assert init["result"]["serverInfo"]["name"] == "starmind-enterprise-mcp"

    tools = cli.list_tools()
    assert any(t["name"] == "lookup_employee" for t in tools)

    call_resp = cli.call_tool("lookup_employee", {"name": "林产品"})
    assert call_resp["result"]["isError"] is False
    parsed = json.loads(call_resp["result"]["content"][0]["text"])
    assert parsed["dept"] == "智能产品部"

    policy = cli.read_resource("starmind://policies/mcp-security")
    assert "最小权限白名单" in policy


def test_mcp_jsonrpc_standard_error_codes():
    srv = create_starmind_enterprise_mcp_server()
    cli = MCPClient(srv.handle_json_line)

    unknown_method = cli.request("tools/delete_all")
    assert unknown_method["error"]["code"] == ERR_METHOD_NOT_FOUND

    missing_param = cli.call_tool("lookup_employee", {})
    assert missing_param["error"]["code"] == ERR_INVALID_PARAMS


def test_long_term_memory_persists_across_sessions(tmp_path: Path):
    db_file = tmp_path / "memory.db"
    ltm1 = SQLiteLongTermMemory(db_file)
    ltm1.remember("emp_01", "preference", "代码评审优先指出并发安全隐患", importance=5)
    ltm1.remember("emp_01", "fact", "所在部门为企业研发二部", importance=3)
    ltm1.close()

    # 模拟新会话重启连接同一 SQLite 文件
    ltm2 = SQLiteLongTermMemory(db_file)
    mgr = HybridMemoryManager("emp_01", ltm2, max_short_messages=2)
    mgr.short_term.add("user", "第一句")
    mgr.short_term.add("assistant", "第一答")
    mgr.short_term.add("user", "帮我做一下并发代码评审")
    ctx = mgr.build_prompt_context("并发代码评审", top_k=1)

    assert len(ctx["short_term_messages"]) == 2
    assert ctx["evicted_count"] == 1
    assert "并发安全隐患" in ctx["long_term_memories"][0]
    ltm2.close()

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 MCP Server 增加 prompts/list 与 prompts/get 原语

**题目要求：**
扩展一个支持 `prompts/list` 与 `prompts/get` 的小型处理器，接收模板名 `weekly_report` 与参数 `dept`，渲染出标准周报提示词。

**思路提示：** 用字典存储模板字符串，调用 `.format(**arguments)` 渲染并返回 `messages` 列表。


In [ ]:
"""Day 44 作业 1 参考答案：MCP Prompts 原语处理器。"""

from typing import Dict

PROMPTS = {
    "weekly_report": "请以【{dept}】负责人视角，按“本周成果/风险阻塞/下周计划”三段式输出周报。"
}


def get_prompt(name: str, arguments: Dict[str, str]) -> Dict[str, object]:
    if name not in PROMPTS:
        raise KeyError(f"未找到提示词模板: {name}")
    text = PROMPTS[name].format(**arguments)
    return {"messages": [{"role": "user", "content": {"type": "text", "text": text}}]}


if __name__ == "__main__":
    res = get_prompt("weekly_report", {"dept": "AI 平台部"})
    print(res)
    assert "AI 平台部" in res["messages"][0]["content"]["text"]


**答案解析与评分标准：**

MCP 的 Prompts 原语让企业可以把优秀 Prompt 模板集中托管在服务端，团队升级模板时所有客户端零发版生效。


### 作业 2（企业实战）：实现基于规则的用户偏好自动提取器

**题目要求：**
编写 `extract_preferences(utterance)`，从用户自然语言（如“以后回复我都用表格”“记住我在杭州研发中心”）中自动识别值得写入长期记忆的条目与类别。

**思路提示：** 匹配“以后/请始终/喜欢用”归类为 `preference`，匹配“我在/我负责”归类为 `fact`。


In [ ]:
"""Day 44 作业 2 参考答案：自然语言长期记忆候选提取器。"""

from typing import List, Tuple


def extract_preferences(utterance: str) -> List[Tuple[str, str, int]]:
    items: List[Tuple[str, str, int]] = []
    if any(k in utterance for k in ("以后", "请始终", "喜欢用")):
        items.append(("preference", utterance.strip(), 5))
    elif any(k in utterance for k in ("我在", "我负责")):
        items.append(("fact", utterance.strip(), 4))
    return items


if __name__ == "__main__":
    p1 = extract_preferences("以后回复我的周报都用表格列出")
    p2 = extract_preferences("今天天气不错")
    print("提取结果:", p1, "| 闲聊过滤:", p2)
    assert len(p1) == 1 and p1[0][0] == "preference"
    assert p2 == []


**答案解析与评分标准：**

先用轻量规则或分类小模型做“记忆门控（Memory Gate）”，能把 90% 的闲聊噪音挡在长期记忆库门外。


### 作业 3（架构进阶）：为长期记忆增加时间衰减（Time Decay）排序因子

**题目要求：**
实现 `decay_score(importance, age_days, half_life_days=30.0)`，结合重要性与记忆距今天数计算衰减分，确保半年前的旧事实权重低于昨天的新事实。

**思路提示：** 使用指数衰减公式 `importance * (0.5 ** (age_days / half_life_days))`。


In [ ]:
"""Day 44 作业 3 参考答案：长期记忆指数时间衰减评分。"""


def decay_score(importance: int, age_days: float, half_life_days: float = 30.0) -> float:
    return round(importance * (0.5 ** (max(age_days, 0.0) / half_life_days)), 4)


if __name__ == "__main__":
    recent = decay_score(importance=4, age_days=1.0)
    stale = decay_score(importance=4, age_days=90.0)
    print("近期记忆分:", recent, "| 90天前记忆分:", stale)
    assert recent > stale and stale == 0.5


**答案解析与评分标准：**

结合相关性（Relevance）、重要性（Importance）与时效衰减（Recency）正是斯坦福 Generative Agents 论文提出的三维记忆检索公式。


---

## 🎯 老陈收尾复盘

今天这一课把 Agent 的“外接手脚（MCP）”和“海马体（长短期记忆）”全部打通了！以后你们在公司推 AI 平台建设，完全可以立一个标准：所有业务部门把自己的能力封装成标准 MCP Server 注册上来，Agent 研发团队只管编排和记忆治理，彻底告别重复造轮子。明天周测 + Dify/Coze 拆解，晚上记得复习一下 ReAct 和 LangGraph！
